### **1. Monthly sales trend (With growth rate)**

In [0]:
%sql
WITH monthly_sales AS (
    SELECT
        YEAR(InvoiceDate) AS yr,
        MONTH(InvoiceDate) AS mn,
        SUM(Quantity * Price) AS revenue
    FROM retail
    GROUP BY yr, mn
)
SELECT
    yr,
    mn,
    revenue,
    LAG(revenue) OVER (ORDER BY yr, mn) AS prev_month_revenue,
    (revenue - LAG(revenue) OVER (ORDER BY yr, mn)) / 
    LAG(revenue) OVER (ORDER BY yr, mn) AS growth_rate
FROM monthly_sales
ORDER BY yr, mn

yr,mn,revenue,prev_month_revenue,growth_rate
2009,12,825685.7600000115,null,null
2010,1,652708.5019999943,825685.7600000115,-0.2094952660925324
2010,2,553713.3060000008,652708.5019999943,-0.1516683108871077
2010,3,833570.130999969,553713.3060000008,0.5054182768726309
2010,4,681528.9919999798,833570.130999969,-0.18239753722652857
2010,5,659858.8599999907,681528.9919999798,-0.0317963465301705
2010,6,752270.1399999794,659858.8599999907,0.14004703975633523
2010,7,650712.939999985,752270.1399999794,-0.13500097185832358
2010,8,697274.9099999849,650712.939999985,0.0715553159277898
2010,9,924333.0109999602,697274.9099999849,0.32563641362053347


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Monthly Sales Conclusion**
1. Strong year-end spike (Oct–Nov peak)
2. Sales drop sharply in early months (Jan–Feb weak demand)
3. Consistent seasonal pattern across years
4. 2011 shows highest peak → demand growth over time
5. December drop in 2011 → incomplete data or anomaly

**Monthly Growth rate cycle Conclusion**
1. Growth is highly volatile → no stable expansion
2. Positive spikes followed by sharp drops → demand shocks
3. Similar patterns across years → recurring seasonal behavior
4. Strong positive growth before peak months → buildup effect
5. Sharp negative in Dec (esp 2011) → anomaly or incomplete data

### **2. Top Products by Revenue (Ranking)**

In [0]:
%sql
SELECT
    Description,
    SUM(Quantity * Price) AS revenue,
    RANK() OVER (ORDER BY SUM(Quantity * Price) DESC) AS rank
FROM retail
GROUP BY Description


-- Below results shows out of 5k products, their are only top 10 products that contributed 6 figure revenue accross all two year span.

Description,revenue,rank
REGENCY CAKESTAND 3 TIER,344563.2500000036,1
Manual,341104.90000000066,2
DOTCOM POSTAGE,322657.48000000016,3
WHITE HANGING HEART T-LIGHT HOLDER,266923.5500000072,4
"PAPER CRAFT , LITTLE BIRDIE",168469.6,5
JUMBO BAG RED RETROSPOT,150935.55999999988,6
PARTY BUNTING,149187.04999999984,7
ASSORTED COLOUR BIRD ORNAMENT,132187.9200000002,8
POSTAGE,127597.42,9
PAPER CHAIN KIT 50'S CHRISTMAS,123141.53999999803,10


### **3. Top Product sold per Month (window + partition)**

In [0]:
%sql
WITH monthly_product AS (
    SELECT
        YEAR(InvoiceDate) AS yr,
        MONTH(InvoiceDate) AS mn,
        Description,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY yr, mn, Description
)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY yr, mn ORDER BY total_qty DESC) AS rnk
    FROM monthly_product
) sub
WHERE rnk = 1

yr,mn,Description,total_qty,rnk
2009,12,WHITE HANGING HEART T-LIGHT HOLDER,6445,1
2010,1,JAZZ HEARTS MEMO PAD,9500,1
2010,2,BLACK AND WHITE PAISLEY FLOWER MUG,19249,1
2010,3,SET/6 WOODLAND PAPER PLATES,13119,1
2010,4,PACK OF 72 RETRO SPOT CAKE CASES,5365,1
2010,5,PACK OF 12 SUKI TISSUES,5625,1
2010,6,WORLD WAR 2 GLIDERS ASSTD DESIGNS,5404,1
2010,7,PACK OF 72 RETRO SPOT CAKE CASES,4493,1
2010,8,SET/6 FRUIT SALAD PAPER CUPS,7131,1
2010,9,BROCADE RING PURSE,13931,1


### **4. Revenue Contribution % of Each Product**

In [0]:
%sql
SELECT
    Description,
    SUM(Quantity * Price) AS revenue,
    SUM(Quantity * Price) / SUM(SUM(Quantity * Price)) OVER () AS revenue_share
FROM retail
GROUP BY Description
ORDER BY revenue DESC

Description,revenue,revenue_share
REGENCY CAKESTAND 3 TIER,344563.2500000036,0.01642892163535514
Manual,341104.90000000066,0.01626402604321733
DOTCOM POSTAGE,322657.48000000016,0.01538444524766096
WHITE HANGING HEART T-LIGHT HOLDER,266923.5500000072,0.012727027869573646
"PAPER CRAFT , LITTLE BIRDIE",168469.6,0.008032701851806882
JUMBO BAG RED RETROSPOT,150935.55999999988,0.007196671401341889
PARTY BUNTING,149187.04999999984,0.007113301704287328
ASSORTED COLOUR BIRD ORNAMENT,132187.9200000002,0.006302775989083498
POSTAGE,127597.42,0.0060838990056353215
PAPER CHAIN KIT 50'S CHRISTMAS,123141.53999999803,0.005871440760780196


### **5. Peak Sales Day of Week**

In [0]:
%sql
SELECT
    DAYOFWEEK(InvoiceDate) AS day,
    SUM(Quantity) AS total_sales
FROM retail
GROUP BY day
ORDER BY total_sales DESC

day,total_sales
5,2377260
3,2173223
2,2057563
4,2054847
6,1698470
1,1053824
7,5119


### **6. Country-wise Top Product**

In [0]:
%sql

WITH country_product AS (
    SELECT
        Country,
        Description,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Country, Description
)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY Country ORDER BY total_qty DESC) AS rnk
    FROM country_product
)
WHERE rnk = 1

Country,Description,total_qty,rnk
Australia,MINI PAINT SET VINTAGE,3060,1
Austria,SET 12 KIDS COLOUR CHALK STICKS,288,1
Bahrain,WHITE TALL PORCELAIN T-LIGHT HOLDER,102,1
Belgium,PACK OF 60 SPACEBOY CAKE CASES,624,1
Bermuda,BOYS ALPHABET IRON ON PATCHES,1152,1
Bermuda,GIRLS ALPHABET IRON ON PATCHES,1152,1
Brazil,DRAGONS BLOOD INCENSE,25,1
Brazil,DOLLY GIRL LUNCH BOX,25,1
Canada,RETRO COFFEE MUGS ASSORTED,504,1
Channel Islands,AFGHAN SLIPPER SOCK PAIR,600,1


### **7. Sales Spike Detection (vs average)**

In [0]:
%sql
WITH daily_sales AS (
    SELECT
        DATE(InvoiceDate) AS dt,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY dt
)
SELECT
    dt,
    total_qty,
    AVG(total_qty) OVER () AS avg_sales,
    CASE 
        WHEN total_qty > 2 * AVG(total_qty) OVER () THEN 'Spike'
        ELSE 'Normal'
    END AS sales_flag
FROM daily_sales

-- Compared each day’s sales against overall average to detect unusually high demand days, Spike logic here is 2X of avg daily sales (quantity)

dt,total_qty,avg_sales,sales_flag
2009-12-01,26204,18907.79139072848,Normal
2009-12-02,31896,18907.79139072848,Normal
2009-12-03,49243,18907.79139072848,Spike
2009-12-04,21325,18907.79139072848,Normal
2009-12-05,5119,18907.79139072848,Normal
2009-12-06,11623,18907.79139072848,Normal
2009-12-07,18058,18907.79139072848,Normal
2009-12-08,23159,18907.79139072848,Normal
2009-12-09,17934,18907.79139072848,Normal
2009-12-10,23718,18907.79139072848,Normal
